In [1]:
import common as cm
import os
import pandas as pd
import numpy as np

# All of US European Performance Matrix

In [3]:
# Folder with pv training rankings
pv_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU")
pv_files = sorted(f for f in os.listdir(pv_dir) if f.endswith(".csv"))
print("Number of files:", len(pv_files))

# Step 1: case count per target ICD, taken from the n_cases column of each file (training half)
case_num = {}
for f in pv_files:
    icd = os.path.splitext(f)[0]
    df = pd.read_csv(os.path.join(pv_dir, f), usecols=["n_cases"])
    case_num[icd] = int(df["n_cases"].iloc[0]) if len(df) > 0 else 0
case_num = pd.Series(case_num)

# Step 2: ICDs with >= 500 cases
CASE_CUTOFF = 500
icd_500 = sorted(case_num[case_num >= CASE_CUTOFF].index)
print("ICDs with n_cases >=", CASE_CUTOFF, ":", len(icd_500), "/", len(case_num))

# Step 3: stack all rankings of the selected target ICDs into one long table
long_list = []
for icd in icd_500:
    df = pd.read_csv(os.path.join(pv_dir, f"{icd}.csv"))
    df["target_icd"] = icd
    long_list.append(df)
long_df = pd.concat(long_list, ignore_index=True)

# Step 4: map PRS phecode to ICD and keep candidate PRS whose ICD is also in the >= 500 set
long_df["prs_icd"] = long_df["prs_phenocode"].map(cm.phecode2icd_map)
long_df = long_df[long_df["prs_icd"].isin(icd_500)]
long_df["method"] = long_df["model_file"].str.extract(r"(C\+T|LDpred2|lassosum2)", expand=False)
long_df["prs_key"] = long_df["prs_icd"] + "_" + long_df["method"]
n_method_per_icd = long_df.groupby("prs_icd")["method"].nunique()
print("Methods per PRS ICD:", n_method_per_icd.value_counts().to_dict())
print("Distinct methods:", sorted(long_df["method"].unique())[:10])

# Step 4b: set AUC of near-discrete PRS models to 0.5 (more than 50% of individuals share one score value)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"],
                  qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))

is_flagged = pd.Series([(p, m) in flagged for p, m in zip(long_df["prs_phenocode"], long_df["model_file"])], index=long_df.index)
long_df.loc[is_flagged, "auc"] = 0.5
print(f"Flagged PRS models (mode share > 0.5): {len(flagged)}; "
      f"rows set to AUC 0.5: {is_flagged.sum()} "
      f"({long_df.loc[is_flagged, 'prs_icd'].nunique()} candidate ICDs)")
print("Flagged rows by method:", long_df.loc[is_flagged, "method"].value_counts().to_dict())

# Step 5: pivot to target ICD x (PRS ICD x method) AUC matrix
aou_auc_full = long_df.pivot_table(index="target_icd", columns="prs_key", values="auc", aggfunc="first")
aou_auc_full = aou_auc_full.reindex(index=icd_500)
aou_auc_full = aou_auc_full.reindex(columns=sorted(aou_auc_full.columns))

# Shape check: expect (n, 3n)
n = len(icd_500)
print("Matrix shape:", aou_auc_full.shape, "expected:", (n, 3 * n))

# Identify PRS ICDs missing one method, and which method is missing
all_methods = {"C+T", "LDpred2", "lassosum2"}
prs_methods = long_df.groupby("prs_icd")["method"].apply(set)
incomplete = prs_methods[prs_methods.apply(len) < 3]
for icd, ms in incomplete.items():
    print(icd, "missing:", all_methods - ms)

Number of files: 428
ICDs with n_cases >= 500 : 428 / 428
Methods per PRS ICD: {3: 424, 2: 4}
Distinct methods: ['C+T', 'LDpred2', 'lassosum2']
Flagged PRS models (mode share > 0.5): 23; rows set to AUC 0.5: 9844 (22 candidate ICDs)
Flagged rows by method: {'C+T': 7704, 'lassosum2': 2140}
Matrix shape: (428, 1280) expected: (428, 1284)
H83 missing: {'C+T'}
I31 missing: {'LDpred2'}
J35 missing: {'lassosum2'}
L01 missing: {'C+T'}


In [4]:
from scipy.stats import skew

# Fixed method order so within_auc / within_rank lists line up across targets
method_order = ["C+T", "LDpred2", "lassosum2"]

# Per-target sample counts (constant within each ranking file)
n_info = long_df.groupby("target_icd")[["n_samples", "n_cases"]].first().astype(int)

summary_rows = []

# Loop over all target traits in the full AUC matrix
for icd in aou_auc_full.index:
    # All candidate AUCs for this target (drop missing), sorted descending
    auc_vec = aou_auc_full.loc[icd].dropna().sort_values(ascending=False)

    # Robust center and scale of the candidate AUC distribution
    center = auc_vec.median()
    sd_mad = 1.4826 * np.median(np.abs(auc_vec - center))

    # Top-1 candidate: split "ICD_method" into ICD and method
    top1_key = auc_vec.index[0]
    top1_icd, top1_method = top1_key.split("_", 1)

    # Within-trait PRS (same ICD as target), in fixed method order; 2 or 3 values
    within_keys = [f"{icd}_{m}" for m in method_order if f"{icd}_{m}" in auc_vec.index]
    within_auc = [round(auc_vec[k], 4) for k in within_keys]
    within_rank = [auc_vec.index.get_loc(k) + 1 for k in within_keys]

    # Collect summary for this target
    summary_rows.append({
        "icd": icd,
        "case": n_info.loc[icd, "n_cases"],
        "control": n_info.loc[icd, "n_samples"] - n_info.loc[icd, "n_cases"],
        "sample": n_info.loc[icd, "n_samples"],
        "chapter": cm.assign_chapter(icd),

        "top1_auc": auc_vec.iloc[0],
        "top1_icd": top1_icd,
        "top1_method": top1_method,

        "median_auc": center,
        "top10_cutoff_auc": auc_vec.iloc[9],

        "within_method": [k.split("_", 1)[1] for k in within_keys],
        "within_auc": within_auc,
        "within_rank": within_rank,

        "skew": skew(auc_vec - center),
        "top1_z_mad": (auc_vec.iloc[0] - center) / sd_mad, # Standardized distance of top1 from the bulk (median / MAD)
    })

# Assemble summary table, sorted by how far the top candidate stands out
aou_auc_summary = pd.DataFrame(summary_rows).reset_index(drop=True)

# Sanity checks
print("Summary shape:", aou_auc_summary.shape)
print(aou_auc_summary.round(4).to_string())

Summary shape: (428, 15)
     icd   case  control  sample                         chapter  top1_auc top1_icd top1_method  median_auc  top10_cutoff_auc              within_method                within_auc         within_rank    skew  top1_z_mad
0    A04   2457   144651  147108             Infectious diseases    0.5374      M51     LDpred2      0.5025            0.5259  [C+T, LDpred2, lassosum2]  [0.4894, 0.5065, 0.5046]    [1256, 377, 501]  0.5527      5.1929
1    A08   1867   145241  147108             Infectious diseases    0.5375      M51     LDpred2      0.5038            0.5276  [C+T, LDpred2, lassosum2]  [0.5103, 0.5191, 0.5145]      [290, 61, 147]  0.3618      4.1227
2    A09   2001   145107  147108             Infectious diseases    0.5214      E06         C+T      0.5015            0.5179  [C+T, LDpred2, lassosum2]   [0.5122, 0.513, 0.4886]      [63, 48, 1239] -0.1108      2.8481
3    A41   7509   139599  147108             Infectious diseases    0.5516      E66     LDpred2    

In [6]:
# Save to file
aou_auc_full.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "aou_auc_landscape_full_matrix.csv"), index=True)
aou_auc_summary.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "aou_auc_landscape_summary.csv"), index=False)

# UK Biobank European Performance Matrix

In [7]:
# Folder with pv training rankings
pv_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "UKB")
pv_files = sorted(f for f in os.listdir(pv_dir) if f.endswith(".csv"))
print("Number of files:", len(pv_files))

# Step 1: case count per target ICD, taken from the n_cases column of each file (training half)
case_num = {}
for f in pv_files:
    icd = os.path.splitext(f)[0]
    df = pd.read_csv(os.path.join(pv_dir, f), usecols=["n_cases"])
    case_num[icd] = int(df["n_cases"].iloc[0]) if len(df) > 0 else 0
case_num = pd.Series(case_num)

# Step 2: ICDs with >= 500 cases
CASE_CUTOFF = 500
icd_500 = sorted(case_num[case_num >= CASE_CUTOFF].index)
print("ICDs with n_cases >=", CASE_CUTOFF, ":", len(icd_500), "/", len(case_num))

# Step 3: stack all rankings of the selected target ICDs into one long table
long_list = []
for icd in icd_500:
    df = pd.read_csv(os.path.join(pv_dir, f"{icd}.csv"))
    df["target_icd"] = icd
    long_list.append(df)
long_df = pd.concat(long_list, ignore_index=True)

# Step 4: map PRS phecode to ICD and keep candidate PRS whose ICD is also in the >= 500 set
long_df["prs_icd"] = long_df["prs_phenocode"].map(cm.phecode2icd_map)
long_df = long_df[long_df["prs_icd"].isin(icd_500)]
long_df["method"] = long_df["model_file"].str.extract(r"(C\+T|LDpred2|lassosum2)", expand=False)
long_df["prs_key"] = long_df["prs_icd"] + "_" + long_df["method"]
n_method_per_icd = long_df.groupby("prs_icd")["method"].nunique()
print("Methods per PRS ICD:", n_method_per_icd.value_counts().to_dict())
print("Distinct methods:", sorted(long_df["method"].unique())[:10])

# Step 4b: set AUC of near-discrete PRS models to 0.5 (more than 50% of individuals share one score value)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_eur.csv"))
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "prs_phenocode"],
                  qc_df.loc[qc_df["flag_mode_0.5"], "model_file"]))

is_flagged = pd.Series([(p, m) in flagged for p, m in zip(long_df["prs_phenocode"], long_df["model_file"])], index=long_df.index)
long_df.loc[is_flagged, "auc"] = 0.5
print(f"Flagged PRS models (mode share > 0.5): {len(flagged)}; "
      f"rows set to AUC 0.5: {is_flagged.sum()} "
      f"({long_df.loc[is_flagged, 'prs_icd'].nunique()} candidate ICDs)")
print("Flagged rows by method:", long_df.loc[is_flagged, "method"].value_counts().to_dict())

# Step 5: pivot to target ICD x (PRS ICD x method) AUC matrix
ukb_auc_full = long_df.pivot_table(index="target_icd", columns="prs_key", values="auc", aggfunc="first")
ukb_auc_full = ukb_auc_full.reindex(index=icd_500)
ukb_auc_full = ukb_auc_full.reindex(columns=sorted(ukb_auc_full.columns))

# Shape check: expect (n, 3n)
n = len(icd_500)
print("Matrix shape:", ukb_auc_full.shape, "expected:", (n, 3 * n))

# Identify PRS ICDs missing one method, and which method is missing
all_methods = {"C+T", "LDpred2", "lassosum2"}
prs_methods = long_df.groupby("prs_icd")["method"].apply(set)
incomplete = prs_methods[prs_methods.apply(len) < 3]
for icd, ms in incomplete.items():
    print(icd, "missing:", all_methods - ms)

Number of files: 368
ICDs with n_cases >= 500 : 368 / 368
Methods per PRS ICD: {3: 365, 2: 3}
Distinct methods: ['C+T', 'LDpred2', 'lassosum2']
Flagged PRS models (mode share > 0.5): 23; rows set to AUC 0.5: 6624 (17 candidate ICDs)
Flagged rows by method: {'C+T': 5520, 'lassosum2': 1104}
Matrix shape: (368, 1101) expected: (368, 1104)
H83 missing: {'C+T'}
I31 missing: {'LDpred2'}
J35 missing: {'lassosum2'}


In [8]:
from scipy.stats import skew

# Fixed method order so within_auc / within_rank lists line up across targets
method_order = ["C+T", "LDpred2", "lassosum2"]

# Per-target sample counts in UKB (constant within each ranking file)
ukb_n_info = long_df.groupby("target_icd")[["n_samples", "n_cases"]].first().astype(int)

summary_rows = []

# Loop over all target traits in the UKB full AUC matrix
for icd in ukb_auc_full.index:
    # All candidate AUCs for this target (drop missing), sorted descending
    auc_vec = ukb_auc_full.loc[icd].dropna().sort_values(ascending=False)

    # Robust center and scale of the candidate AUC distribution
    center = auc_vec.median()
    sd_mad = 1.4826 * np.median(np.abs(auc_vec - center))

    # Top-1 candidate: split "ICD_method" into ICD and method
    top1_key = auc_vec.index[0]
    top1_icd, top1_method = top1_key.split("_", 1)

    # Within-trait PRS (same ICD as target), in fixed method order; 2 or 3 values
    within_keys = [f"{icd}_{m}" for m in method_order if f"{icd}_{m}" in auc_vec.index]
    within_auc = [round(auc_vec[k], 4) for k in within_keys]
    within_rank = [auc_vec.index.get_loc(k) + 1 for k in within_keys]

    # Collect summary for this target
    summary_rows.append({
        "icd": icd,
        "case": ukb_n_info.loc[icd, "n_cases"],
        "control": ukb_n_info.loc[icd, "n_samples"] - ukb_n_info.loc[icd, "n_cases"],
        "sample": ukb_n_info.loc[icd, "n_samples"],
        "chapter": cm.assign_chapter(icd),

        "top1_auc": auc_vec.iloc[0],
        "top1_icd": top1_icd,
        "top1_method": top1_method,

        "median_auc": center,
        "top10_cutoff_auc": auc_vec.iloc[9],

        "within_method": [k.split("_", 1)[1] for k in within_keys],
        "within_auc": within_auc,
        "within_rank": within_rank,

        "skew": skew(auc_vec - center),
        "top1_z_mad": (auc_vec.iloc[0] - center) / sd_mad, # Standardized distance of top1 from the bulk (median / MAD)
    })

# Assemble UKB summary table, sorted by how far the top candidate stands out
ukb_auc_summary = pd.DataFrame(summary_rows).reset_index(drop=True)

# Sanity checks
print("UKB summary shape:", ukb_auc_summary.shape)
print(ukb_auc_summary.round(4).to_string())

UKB summary shape: (368, 15)
     icd    case  control  sample                         chapter  top1_auc top1_icd top1_method  median_auc  top10_cutoff_auc              within_method                within_auc        within_rank     skew  top1_z_mad
0    A04    5188   406949  412137             Infectious diseases    0.5294      M51     LDpred2      0.5028            0.5218  [C+T, LDpred2, lassosum2]   [0.5007, 0.509, 0.5087]    [731, 171, 184]   0.6385      5.2217
1    A08    2085   410052  412137             Infectious diseases    0.5361      F32     LDpred2      0.5022            0.5252  [C+T, LDpred2, lassosum2]  [0.5186, 0.5231, 0.5284]        [38, 19, 7]   0.5747      4.7739
2    A09   21683   390454  412137             Infectious diseases    0.5285      M79     LDpred2      0.5023            0.5191  [C+T, LDpred2, lassosum2]  [0.5099, 0.5193, 0.4999]       [90, 9, 871]   1.3338      7.7353
3    A41   16565   395572  412137             Infectious diseases    0.5224      E66     LD

In [9]:
# Save to file
ukb_auc_full.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "ukb_auc_landscape_full_matrix.csv"), index=True)
ukb_auc_summary.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "ukb_auc_landscape_summary.csv"), index=False)

# Process AFR Performance matrix

In [10]:
# generate mapping between phecode and icd
mapping = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "african_icd_phenocode_pairs.csv"))[['icd', 'phecode']] 
mapping['phecode'] = 'Phe' + mapping['phecode'].astype(str).apply(
    lambda x: x.split('.')[0].zfill(3) + ('.' + x.split('.')[1] if '.' in x else '')
)
phe_to_icd = dict(zip(mapping['phecode'], mapping['icd']))

In [11]:
# Folder with pv training rankings
pv_dir = os.path.join(cm.ROOT_PATH, "prs_ranking", "AoU_AFR")
pv_files = sorted(f for f in os.listdir(pv_dir) if f.endswith(".csv"))
print("Number of files:", len(pv_files))

# Step 1: case count per target ICD, taken from the n_cases column of each file (training half)
case_num = {}
for f in pv_files:
    icd = os.path.splitext(f)[0]
    df = pd.read_csv(os.path.join(pv_dir, f), usecols=["n_cases"])
    case_num[icd] = int(df["n_cases"].iloc[0]) if len(df) > 0 else 0
case_num = pd.Series(case_num)

# Step 2: ICDs with >= 500 cases
CASE_CUTOFF = 500
icd_500 = sorted(case_num[case_num >= CASE_CUTOFF].index)
print("ICDs with n_cases >=", CASE_CUTOFF, ":", len(icd_500), "/", len(case_num))

# Step 3: stack all rankings of the selected target ICDs into one long table
long_list = []
for icd in icd_500:
    df = pd.read_csv(os.path.join(pv_dir, f"{icd}.csv"))
    df["target_icd"] = icd
    long_list.append(df)
long_df = pd.concat(long_list, ignore_index=True)

# Step 4: map PRS phecode to ICD (strip ".sscore" suffix first) and keep candidates in the >= 500 set
long_df["prs_phecode"] = long_df["prs_trait"].str.replace(r"\.sscore$", "", regex=True)
long_df["prs_icd"] = long_df["prs_phecode"].map(phe_to_icd)
print("Unmapped PRS rows:", long_df["prs_icd"].isna().sum(),
      sorted(long_df.loc[long_df["prs_icd"].isna(), "prs_phecode"].unique())[:10])
long_df = long_df[long_df["prs_icd"].isin(icd_500)]

# Step 4a: harmonize method labels with the QC table (C_T -> C+T)
long_df["method"] = long_df["method"].replace({"C_T": "C+T"})

# Step 4b: set AUC of near-discrete PRS models to 0.5 (more than 50% of individuals share one score value)
# File names repeat across method folders, so match on (method, phecode)
qc_df = pd.read_csv(os.path.join(cm.ROOT_PATH, "reference_data", "prs_model_qc_afr.csv"))
qc_df["prs_phecode"] = qc_df["model_file"].str.replace(r"\.sscore$", "", regex=True)
flagged = set(zip(qc_df.loc[qc_df["flag_mode_0.5"], "method"],
                  qc_df.loc[qc_df["flag_mode_0.5"], "prs_phecode"]))

is_flagged = pd.Series([(m, p) in flagged for m, p in zip(long_df["method"], long_df["prs_phecode"])],
                       index=long_df.index)
long_df.loc[is_flagged, "auc"] = 0.5
print(f"Flagged PRS models (mode share > 0.5): {len(flagged)}; "
      f"rows set to AUC 0.5: {is_flagged.sum()} "
      f"({long_df.loc[is_flagged, 'prs_icd'].nunique()} candidate ICDs)")
print("Flagged rows by method:", long_df.loc[is_flagged, "method"].value_counts().to_dict())

# Method already provided as a column (C_T / LDpred2 / lassosum2)
long_df["prs_key"] = long_df["prs_icd"] + "_" + long_df["method"]
n_method_per_icd = long_df.groupby("prs_icd")["method"].nunique()
print("Methods per PRS ICD:", n_method_per_icd.value_counts().to_dict())
print("Distinct methods:", sorted(long_df["method"].unique()))

# Step 5: pivot to target ICD x (PRS ICD x method) AUC matrix
afr_auc_full = long_df.pivot_table(index="target_icd", columns="prs_key", values="auc", aggfunc="first")
afr_auc_full = afr_auc_full.reindex(index=icd_500)
afr_auc_full = afr_auc_full.reindex(columns=sorted(afr_auc_full.columns))

# Shape check: expect (n, 3n)
n = len(icd_500)
print("Matrix shape:", afr_auc_full.shape, "expected:", (n, 3 * n))

# Identify PRS ICDs missing one method, and which method is missing
all_methods = {"C+T", "LDpred2", "lassosum2"}
prs_methods = long_df.groupby("prs_icd")["method"].apply(set)
incomplete = prs_methods[prs_methods.apply(len) < 3]
for icd, ms in incomplete.items():
    print(icd, "missing:", all_methods - ms)

Number of files: 138
ICDs with n_cases >= 500 : 138 / 138
Unmapped PRS rows: 0 []
Flagged PRS models (mode share > 0.5): 12; rows set to AUC 0.5: 1656 (9 candidate ICDs)
Flagged rows by method: {'C+T': 1242, 'lassosum2': 414}
Methods per PRS ICD: {3: 138}
Distinct methods: ['C+T', 'LDpred2', 'lassosum2']
Matrix shape: (138, 414) expected: (138, 414)


In [12]:
from scipy.stats import skew

# Fixed method order so within_auc / within_rank lists line up across targets
method_order = ["C+T", "LDpred2", "lassosum2"]

# Per-target sample counts in UKB (constant within each ranking file)
afr_n_info = long_df.groupby("target_icd")[["n_samples", "n_cases"]].first().astype(int)

summary_rows = []

# Loop over all target traits in the UKB full AUC matrix
for icd in afr_auc_full.index:
    # All candidate AUCs for this target (drop missing), sorted descending
    auc_vec = afr_auc_full.loc[icd].dropna().sort_values(ascending=False)

    # Robust center and scale of the candidate AUC distribution
    center = auc_vec.median()
    sd_mad = 1.4826 * np.median(np.abs(auc_vec - center))

    # Top-1 candidate: split "ICD_method" into ICD and method
    top1_key = auc_vec.index[0]
    top1_icd, top1_method = top1_key.split("_", 1)

    # Within-trait PRS (same ICD as target), in fixed method order; 2 or 3 values
    within_keys = [f"{icd}_{m}" for m in method_order if f"{icd}_{m}" in auc_vec.index]
    within_auc = [round(auc_vec[k], 4) for k in within_keys]
    within_rank = [auc_vec.index.get_loc(k) + 1 for k in within_keys]

    # Collect summary for this target
    summary_rows.append({
        "icd": icd,
        "case": afr_n_info.loc[icd, "n_cases"],
        "control": afr_n_info.loc[icd, "n_samples"] - afr_n_info.loc[icd, "n_cases"],
        "sample": afr_n_info.loc[icd, "n_samples"],
        "chapter": cm.assign_chapter(icd),

        "top1_auc": auc_vec.iloc[0],
        "top1_icd": top1_icd,
        "top1_method": top1_method,

        "median_auc": center,
        "top10_cutoff_auc": auc_vec.iloc[9],

        "within_method": [k.split("_", 1)[1] for k in within_keys],
        "within_auc": within_auc,
        "within_rank": within_rank,

        "skew": skew(auc_vec - center),
        "top1_z_mad": (auc_vec.iloc[0] - center) / sd_mad, # Standardized distance of top1 from the bulk (median / MAD)
    })

# Assemble UKB summary table, sorted by how far the top candidate stands out
afr_auc_summary = pd.DataFrame(summary_rows).reset_index(drop=True)

# Sanity checks
print("AFR summary shape:", afr_auc_summary.shape)
print(afr_auc_summary.round(4).to_string())

AFR summary shape: (138, 15)
     icd   case  control  sample                         chapter  top1_auc top1_icd top1_method  median_auc  top10_cutoff_auc              within_method                within_auc      within_rank    skew  top1_z_mad
0    B35   5742    41122   46864             Infectious diseases    0.5203      E11   lassosum2      0.5016            0.5125  [C+T, LDpred2, lassosum2]   [0.5082, 0.5182, 0.506]      [39, 3, 63]  0.2597      4.3698
1    B36    603    46261   46864             Infectious diseases    0.5500      R79     LDpred2      0.5016            0.5275  [C+T, LDpred2, lassosum2]  [0.5178, 0.5265, 0.5227]     [53, 14, 29]  0.2042      3.4030
2    C50   1162    45702   46864                       Neoplasms    0.5250      R16   lassosum2      0.4989            0.5170  [C+T, LDpred2, lassosum2]  [0.5105, 0.5223, 0.4888]     [42, 3, 353]  0.0828      2.5552
3    D25   3545    26469   30014                       Neoplasms    0.5171      D62   lassosum2      0.5008

In [13]:
# Save to file
afr_auc_full.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "afr_auc_landscape_full_matrix.csv"), index=True)
afr_auc_summary.to_csv(os.path.join(cm.ROOT_PATH, "result", "auc_landscape", "afr_auc_landscape_summary.csv"), index=False)